In [19]:
import pandas as pd
import numpy as np
import seaborn as sns

print("pandas:", pd.__version__, "numpy:", np.__version__, "seaborn:", sns.__version__)

pandas: 3.1.1 numpy: 2.5.5 seaborn: 0.13.2


In [20]:
df = sns.load_dataset('titanic')
df.head(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [21]:
df.shape

(891, 15)

In [22]:
# 자료형 조회
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [23]:
df.describe() # 기초 통계량을 한 번에 요약해 주는 함수

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [ ]:
# 각 셀의 널(NaN) 여부 ( NaN : Not a number)
df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [ ]:
# 나이가 비어있는 승객 수, 성별, 클래스                                            
empty_age = df["age"].isnull()
print(f"나이 빈칸: {empty_age.sum()}명")    
df.loc[empty_age, ["sex", "age", "class"] ].head()

In [ ]:
# 평균값, 중앙값 산출
print("평균값: ", round(df["age"].mean(), 2))
print("중앙값: ", round(df["age"].median(), 2))

mean_age = df["age"].mean()
# fillna : 빈칸(NaN)만 주어진 값을 새로운 값으로 채우는 메소드
df.loc[:, "age"] = df["age"].fillna(mean_age)

# df["age"].isnull().sum()
df.loc[empty_age, ["sex", "age", "class"] ].head()

In [ ]:
# 탑승 할구 : 최빈값 → 데이터에서 가장 많이 나타나는 값
df["embark_town"].value_counts(dropna=False)

In [ ]:
# 최빈값 추출 후 빈칸에 채우는 로직
mode_town = df["embark_town"].mode()[0]
print(f"최빈값 : {mode_town}")

df.loc[:, "embark_town"] = df["embark_town"].fillna(mode_town)
df["embark_town"].value_counts(dropna=False)

In [ ]:
# embarked town값으로 설정
df.loc[:, "embarked"] = df["embark_town"].str[0]
df[["embarked","embark_town"]].isnull().sum()

In [ ]:
# print(df["deck"].dtype)
# df["deck"].cat.categories

df["deck"] = df["deck"].cat.add_categories("Unknown").fillna("Unknown")

print(df["deck"].cat.categories.to_list())
print(df["deck"].value_counts())

In [ ]:
df.isnull().sum()

In [ ]:
# 튀는 값 확인

print(df["fare"].quantile([0.25, 0.5, 0.75]))
df["fare"].describe()[["25%", "50%", "75%", ]]

# IQR 계산 (Q3 - Q1)
# Q3 + 1.5 * IQR
q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr # 통계학적으로 이상수치를 발견했을 때 가장 흔히 쓰는 관례적인 배수이다

print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f} UPPER={upper:.2f}")
print("이상 승선요금:", (df["fare"] > upper).sum(), "명")

In [ ]:
# fare(요금)가 512보다 큰 행만 골라서 보여주는 코드
df[ df["fare"] >= 512]

In [ ]:
before = len(df)
df = df[ df["fare"] < 512].copy()
print(before, "=>", len(df))

In [ ]:
# 중복되는 행 확인
print("중복 행:", df.duplicated().sum(), "개")
# df[조건]
df[df.duplicated(keep=False)].sort_values(["fare", "age"]).head(5)  # True는 첫번째나오는 것은 중복이 아니라고 체크하고 다음에 나오는 걸 중복이라고 체크함

In [ ]:
import platform
import matplotlib.pyplot as plts

os_name = platform.system()

if os_name == "Window":
    plts.rcParams["font.family"] = "Malgun Gothic"

# 나이 분포 차트
fig, ax = plts.subplots(figsize=(6, 4))
ax.hist(df["age"], bins=20, color="blue", edgecolor="black")
ax.set_title("나이분포")
ax.set_xlabel("나이(세)")